# Exploit pack (play-a-hand vs archetype) — GPU on Kaggle

Re-solves the **exploit** content (full hand vs a Calling Station / Nit / Maniac / LAG / Reg,
graded on the hero's multi-street best response to the pinned archetype) and writes the signed
`exploit_v1` pack.

Runs on the **batched GPU solver** (`--solver gpu`): one GTO solve per board is reused, then
each (archetype, seat) is a cheap eval pass with the villain pinned and the hero best-responding.
The extraction is the same correct streets=3 engine as continuation (independent-oracle verified).

Pick a **GPU** session; **Internet On** (to clone the repo).

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout or 'NO GPU — switch the runtime to GPU')

In [ ]:
!rm -rf /kaggle/working/poker && git clone -q --depth 1 https://github.com/tian-chaiyaporn2/poker_offline_trainer /kaggle/working/poker
import sys; sys.path.insert(0, '/kaggle/working/poker/src')
import subprocess
print('source ready @', subprocess.run(['git','-C','/kaggle/working/poker','rev-parse','--short','HEAD'],
                                       capture_output=True, text=True).stdout.strip())

In [ ]:
# Solve the exploit library on GPU and write the signed pack.
#   FLOPS: curated runouts (max 4). N: combos/range. ITERS: CFR iterations (stable per flop).
#   All 5 archetypes are generated per board; reg = best response vs GTO.
import subprocess, os
FLOPS, N, ITERS, VERSION = 4, 80, 600, 'exploit_v1'
env = {**os.environ, 'PYTHONPATH': 'src'}
subprocess.run(
    ['python', 'demo/gen_exploit.py', '--solver', 'gpu',
     '--flops', str(FLOPS), '--n', str(N), '--iters', str(ITERS), '--version', VERSION],
    cwd='/kaggle/working/poker', env=env, check=True)

In [ ]:
import shutil, os
VERSION = 'exploit_v1'
base = '/kaggle/working/poker/output/packs'
files = [f'flop_pack_{VERSION}.db', f'flop_pack_{VERSION}.db.gz', f'build_report_{VERSION}.json']
if not os.path.exists(os.path.join(base, files[0])):
    raise SystemExit('No pack written -- check the solve cell for errors.')
for f in files:
    shutil.copy(os.path.join(base, f), os.path.join('/kaggle/working', f))
print('DOWNLOAD from /kaggle/working:')
for f in files:
    print('  %-42s %d KB' % (f, os.path.getsize(os.path.join('/kaggle/working', f)) // 1024))